# Task 1: Parameter-Efficient Fine-Tuning (LoRA) on Qwen2.5-0.5B-Instruct
**Tndrr AI Internship Assignment – Task 1**

### Objective
Fine-tune an open-weight language model (< 1.5B parameters) to convert natural-language user instructions into a strict, predefined JSON schema using LoRA/QLoRA on Google Colab (free T4 GPU).

### Key Specifications
- **Base Model:** `Qwen/Qwen2.5-0.5B-Instruct` (494M parameters, Apache 2.0)
- **PEFT Method:** LoRA (r=8, alpha=16, dropout=0.05, targets: `q_proj`, `k_proj`, `v_proj`, `o_proj`)
- **Dataset:** 300 training examples, 60 held-out test examples across 6 balanced intents
- **Target Schema:** `{"intent": "<string>", "entities": {<canonical keys>}}`
- **Deterministic Evaluation:** Greedy decoding (`temperature=0.0`, `do_sample=False`)
- **Evaluation Metrics:** JSON validity rate, Exact-match accuracy, Intent accuracy, Entity key accuracy


## 1. Environment & Hardware Verification
Verify that a GPU is available. In Google Colab, select **Runtime > Change runtime type > T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device     :', torch.cuda.get_device_name(0))
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Total VRAM     : {vram_gb:.2f} GB')


## 2. Install Required Dependencies
Install HuggingFace ecosystem packages with exact version compatibility.

In [ ]:
# Uninstall pre-installed incompatible torchao on Colab then install required packages
!pip uninstall -y torchao
!pip install -q torch transformers peft datasets accelerate bitsandbytes scipy


## 3. Clone Repository or Navigate to Directory
Ensure current working directory is `task1_lora/` containing all configuration and scripts.

In [ ]:
import os
from pathlib import Path

# Clone latest repo if not present, or pull latest changes
if not Path('/content/Tndrr-ai').exists():
    !git clone https://github.com/Rahul0Developer/Tndrr-ai.git
else:
    !cd /content/Tndrr-ai && git pull

# Switch directory to task1_lora
%cd /content/Tndrr-ai/tndrr-ai-internship-assignment/task1_lora

print('Current directory:', os.getcwd())
!ls -la


## 4. Dataset Generation and Strict Schema Validation
Generates 300 train and 60 test examples with 6 balanced intents and zero split leakage, then validates all schema invariants.

In [ ]:
!python create_dataset.py
!python validate_dataset.py


## 5. Execute LoRA / QLoRA Fine-Tuning
Trains `Qwen/Qwen2.5-0.5B-Instruct` using PEFT LoRA.
- Target modules: `q_proj`, `k_proj`, `v_proj`, `o_proj`
- Train loss masked strictly to assistant tokens
- Records execution telemetry to `results/train_log.json`
- Saves trained adapter weights to `lora_adapter/`

In [ ]:
# Standard LoRA (fp16 on T4 GPU):
!python train_lora.py

# Or run 4-bit QLoRA if desired:
# !python train_lora.py --qlora


## 6. Base Model vs. Fine-Tuned Model Evaluation
Runs deterministic greedy decoding on the 60 held-out test examples.
Computes JSON validity, exact match, intent accuracy, entity key accuracy, and field-level accuracy.

In [ ]:
!python evaluate.py --eval-mode both


## 7. Results Inspection & Failure Case Analysis
Displays the comparison table and detailed error diagnostics.

In [ ]:
import json
from pathlib import Path

eval_path = Path('results/evaluation_results.json')
if eval_path.exists():
    res = json.loads(eval_path.read_text(encoding='utf-8'))
    print('=' * 65)
    print(f'{"METRIC":<25} | {"BASE MODEL":<15} | {"FINE-TUNED LORA":<15}')
    print('-' * 65)
    base = res.get('base_model', {})
    ft = res.get('fine_tuned_lora', {})
    for key in ['exact_match_accuracy', 'json_validity_rate', 'intent_accuracy', 'entity_key_accuracy', 'field_level_accuracy']:
        label = key.replace('_', ' ').title()
        b_val = f"{base.get(key, 0.0):.1%}"
        f_val = f"{ft.get(key, 0.0):.1%}"
        print(f'{label:<25} | {b_val:<15} | {f_val:<15}')
    print('=' * 65)
    
    failures = res.get('failure_cases', [])
    print(f'\nGENUINE FAILURE EXAMPLES ({len(failures)} cases documented):')
    for idx, fc in enumerate(failures, 1):
        print(f'\n[Case {idx}] Source: {fc.get("source", "N/A")} | Type: {fc.get("failure_type", "Error")}')
        print(f'  Input    : {fc["input"]}')
        print(f'  Expected : {fc["expected_output"]}')
        print(f'  Actual   : {fc["actual_output"]}')
        print(f'  Analysis : {fc["explanation"]}')
else:
    print('results/evaluation_results.json not found.')


## 8. Export Adapter, Merge Weights, and Package Submission
Packages adapter weights into `task1_adapter.zip` and optionally exports a merged standalone model.

In [ ]:
!python export_adapter.py --package-zip task1_adapter.zip
# Optional: merge adapter weights into base model
# !python export_adapter.py --merge --output-dir ./merged_qwen_0.5b
